In [2]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Define mappings
sectors = {
    'Financials': ['Banks', 'Insurance', 'Asset Management'],
    'Technology': ['Software', 'Hardware', 'Semiconductors'],
    'Energy': ['Oil & Gas', 'Midstream', 'Renewables'],
    'Healthcare': ['Pharma', 'Biotech', 'Medical Devices'],
    'Industrials': ['Manufacturing', 'Transportation', 'Aerospace'],
    'Consumer': ['Retail', 'Consumer Products', 'Food & Beverage']
}

countries = ['US', 'UK', 'Germany', 'France', 'Canada', 'Japan', 'Australia']
ratings = ['AAA', 'AA', 'A', 'BBB', 'BB', 'B', 'CCC']

# Rating distribution: 10% AAA, 15% AA, 20% A, 25% BBB, 20% BB, 8% B, 2% CCC
n = 150
rating_counts = {
    'AAA': 15,
    'AA': 22,
    'A': 30,
    'BBB': 38,
    'BB': 30,
    'B': 12,
    'CCC': 3
}

# Rating-based ranges
spread_ranges = {
    'AAA': (20, 60),
    'AA': (40, 90),
    'A': (70, 150),
    'BBB': (120, 250),
    'BB': (250, 450),
    'B': (450, 700),
    'CCC': (700, 1200)
}

yield_ranges = {
    'AAA': (3.5, 4.5),
    'AA': (4.0, 5.0),
    'A': (4.5, 5.5),
    'BBB': (5.0, 6.5),
    'BB': (6.0, 8.5),
    'B': (8.0, 11.0),
    'CCC': (11.0, 16.0)
}

# Generate issuer names
issuer_prefixes = ['Global', 'United', 'First', 'Prime', 'Metro', 'National', 'Pacific', 'Atlantic',
                   'Continental', 'Northern', 'Southern', 'Eastern', 'Western', 'Central', 'International',
                   'Royal', 'Imperial', 'Standard', 'Advanced', 'Dynamic', 'Strategic', 'Premier',
                   'Heritage', 'Liberty', 'Summit', 'Vanguard', 'Pioneer', 'Alliance', 'Consolidated',
                   'Meridian', 'Crest', 'Apex', 'Vertex', 'Horizon', 'Zenith']
issuer_suffixes = ['Corp', 'Inc', 'Holdings', 'Group', 'Capital', 'Financial', 'Industries',
                   'International', 'Systems', 'Energy', 'Technologies', 'Pharma', 'Healthcare',
                   'Manufacturing', 'Resources', 'Partners', 'Ventures', 'Solutions', 'Enterprises']

def generate_issuer():
    return f"{np.random.choice(issuer_prefixes)} {np.random.choice(issuer_suffixes)}"

# Build portfolio
portfolio = []
issuer_pool = set()

for rating, count in rating_counts.items():
    for i in range(count):
        issuer = generate_issuer()
        while issuer in issuer_pool and len(issuer_pool) < 200:
            issuer = generate_issuer()
        issuer_pool.add(issuer)

        sector = np.random.choice(list(sectors.keys()))
        industry = np.random.choice(sectors[sector])
        country = np.random.choice(countries)

        duration = np.random.uniform(1, 10)
        maturity_years = np.random.uniform(duration + 1, 20)

        spread_low, spread_high = spread_ranges[rating]
        spread_bps = np.random.uniform(spread_low, spread_high)

        yield_low, yield_high = yield_ranges[rating]
        yield_val = np.random.uniform(yield_low, yield_high)

        price = np.random.uniform(85, 110)
        position = np.random.uniform(500000, 20000000)

        daily_return = np.random.normal(0.0005, 0.008)
        benchmark_return = np.random.normal(0.0004, 0.007)

        bond_id = f"BOND_{len(portfolio)+1:04d}"

        portfolio.append({
            'Bond_ID': bond_id,
            'Issuer': issuer,
            'Sector': sector,
            'Industry': industry,
            'Rating': rating,
            'Country': country,
            'Position': round(position, 2),
            'Price': round(price, 4),
            'Duration': round(duration, 2),
            'Spread_bps': round(spread_bps, 2),
            'Yield': round(yield_val, 4),
            'Maturity_Years': round(maturity_years, 2),
            'Daily_Return': round(daily_return, 6),
            'Benchmark_Return': round(benchmark_return, 6)
        })

# Create DataFrame
df = pd.DataFrame(portfolio)

# Shuffle to mix ratings
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df['Bond_ID'] = [f"BOND_{i+1:04d}" for i in range(len(df))]

# Calculate Market_Value
df['Market_Value'] = df['Position'] * df['Price'] / 100

# Calculate Portfolio_Weight
total_market_value = df['Market_Value'].sum()
df['Portfolio_Weight'] = df['Market_Value'] / total_market_value

# Generate Benchmark_Weights that sum to 100%
raw_weights = np.random.exponential(1, len(df))
benchmark_weights = raw_weights / raw_weights.sum()
df['Benchmark_Weight'] = benchmark_weights

# Output
print("=" * 80)
print("HEAD OF DATAFRAME")
print("=" * 80)
print(df.head(10).to_string())

print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
numeric_cols = ['Position', 'Price', 'Duration', 'Spread_bps', 'Yield', 'Maturity_Years',
                'Daily_Return', 'Benchmark_Return', 'Market_Value', 'Portfolio_Weight', 'Benchmark_Weight']
print(df[numeric_cols].describe().T.to_string())

print("\n" + "=" * 80)
print("SECTOR DISTRIBUTION")
print("=" * 80)
print(df['Sector'].value_counts().sort_index().to_string())

print("\n" + "=" * 80)
print("RATING DISTRIBUTION")
print("=" * 80)
rating_order = ['AAA', 'AA', 'A', 'BBB', 'BB', 'B', 'CCC']
print(df['Rating'].value_counts().reindex(rating_order).to_string())

print("\n" + "=" * 80)
print("PORTFOLIO SUMMARY")
print("=" * 80)
print(f"Total Bonds: {len(df)}")
print(f"Total Market Value: ${df['Market_Value'].sum():,.2f}")
print(f"Average Price: {df['Price'].mean():.4f}")
print(f"Average Duration: {df['Duration'].mean():.2f} years")
print(f"Average Yield: {df['Yield'].mean():.4f}%")
print(f"Average Spread: {df['Spread_bps'].mean():.2f} bps")
print(f"Weighted Average Yield: {(df['Yield'] * df['Portfolio_Weight']).sum():.4f}%")
print(f"Weighted Average Duration: {(df['Duration'] * df['Portfolio_Weight']).sum():.2f} years")
print(f"Benchmark_Weight sum: {df['Benchmark_Weight'].sum():.6f}")
print(f"Portfolio_Weight sum: {df['Portfolio_Weight'].sum():.6f}")

# Save to CSV
df.to_csv('credit_portfolio_150_bonds.csv', index=False)
print("\nDataFrame saved to 'credit_portfolio_150_bonds.csv'")

HEAD OF DATAFRAME
     Bond_ID                  Issuer       Sector           Industry Rating    Country     Position     Price  Duration  Spread_bps   Yield  Maturity_Years  Daily_Return  Benchmark_Return  Market_Value  Portfolio_Weight  Benchmark_Weight
0  BOND_0001     Standard Industries   Technology     Semiconductors    BBB         US  17907098.37   95.5356      2.87      143.59  5.8746            4.30     -0.003682          0.007743  1.710765e+07          0.011304          0.002155
1  BOND_0002   Premier Manufacturing     Consumer  Consumer Products     AA      Japan  11749625.75   97.5659      4.54       71.56  4.7948           18.44     -0.010751          0.000158  1.146363e+07          0.007575          0.017750
2  BOND_0003            Vanguard Inc   Technology           Hardware     BB  Australia  14899759.58   99.1754      9.79      424.15  7.9560           14.54     -0.001453          0.007149  1.477690e+07          0.009764          0.007590
3  BOND_0004      Global Enter